# Práctica 3 - Ejercicio 1

Asignatura: Programación para la Inteligencia Artificial

Alumno: Fernández Roldán, Daniel

Típicamente el Aprendizjae Profundo se utiliza en problemas supervisados (conocemos la entrada y la salida para los datos de entrenamiento, validación y test). Este planteamiento tiene el problema de requerir que los datos hayan sido previamente etiquetados. Sin embargo, hay modelos neuronales que no tienen esta restricción.

Un Autocodificador (*Autoencoder*) es una red neuronal que se entrena para aprender la función identidad. Típicamente se puede dividir en dos secciones simétricas: una primera parte que codifica progresivamente la entrada a un espacio de menos dimensiones (denominado espacio latente) y una segunda parte que decodifica de vuelta al espacio de dimensiones original.

$x = d_{\theta_{1}}(e_{\theta_{2}}(x))$

Como aprende la función identidad, no requiere etiquetado para su entrenamiento.

Un Autocodificador con eliminación de ruido (*Denoising Autoencoder*) es una red neuronal que se entrena para eliminar el ruido de unos datos de entrada.

$x = d_{\theta_{1}}(e_{\theta_{2}}(\hat{x}))$

El objetivo de esta práctica es definir y entrenar un Autocodificar con eliminación de ruido capaz de lidiar con ruido gaussiano de media $\mu=0$ y desviación típica $\sigma=0.2$ para el conjunto de datos Fashion-MNIST (https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html). El autocodificador debe componerse de neuronas lineales y el espacio latente debe ser de 8 dimensiones.

Para evaluar la efectividad del modelo entrenado se deben utilizar todos los ejemplos del conjunto de test y el error absoluto. Se recomienda incluir el código necesario para mostrar un ejemplo cualquiera del conjunto de test con ruido, sin ruido y el resultado del autocodificador.

Una vez entrenado, se debe mostrar la relación entre las componentes de las codificaciones del conjunto de test con la clase asociada a dichas codificaciones. Se recomienda hacer gráficas 2D cuyos ejes muestren los valores de 2 componentes y muestren la clase como el color de cada punto.

El cuaderno entregado debe llamarse ApellidosNombrePractica3Ejercicio1.ipynb

Ej1: Autoencoder que quite el ruido (test error abs)
Ej2: Autoencoder que genere imágenes nuevas

In [1]:
# Import libraries.
import torch
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from tqdm import tqdm

import matplotlib.pyplot as plt

In [2]:
# Define the transform to convert images to tensors.
transform = transforms.ToTensor()


# Load the Fashion-MNIST dataset.
train_images = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_images = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)

# Split the training dataset into training and validation datasets.
train_size = int(0.8 * len(train_images))
val_size = len(train_images) - train_size

# Define the training, validation, and test datasets.
train_dataset, val_dataset = random_split(train_images, [train_size, val_size])
test_dataset = test_images

# Define the data loaders for training, validation, and testing.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 1.29MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 9.00MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 931kB/s]


In [3]:
def add_noise(image, deviation=0.2):
    """
    Function to add noise to the images in the dataset. 
    The noise is generated from a normal distribution with mean 0 and standard deviation 0.2. 
    The noisy images are clipped to be in the range [0, 1].
    """
    # Generate noise from a normal distribution with mean 0 and standard deviation 0.2.
    noise = torch.randn_like(image) * deviation

    # Add the noise to the image and clip the values to be in the range [0, 1].
    noisy_image = image + noise

    # Clip the values to be in the range [0, 1].
    final_image = torch.clamp(noisy_image, min=0., max=1.)

    return final_image

In [ ]:
class Autoencoder(nn.Module):
    """
    Autoencoder class that defines the architecture of the autoencoder model.
    """
    def __init__(self):
        super(Autoencoder, self).__init__()
        # Define the encoder part of the autoencoder.
        self.encoder = nn.Sequential(
            nn.Flatten(),  # Flatten the input image to a vector.
            nn.Linear(28 * 28, 128),  # Fully connected layer with 128 neurons.
            nn.ReLU(),  # ReLU activation function.
            nn.Linear(128, 64),  # Fully connected layer with 64 neurons.
            nn.ReLU(),  # ReLU activation function.
            nn.Linear(64, 8),  # Fully connected layer with 8 neurons.
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 64),  # Fully connected layer with 64 neurons.
            nn.ReLU(),  # ReLU activation function.
            nn.Linear(64, 128),  # Fully connected layer with 128 neurons.
            nn.ReLU(),  # ReLU activation function.
            nn.Linear(128, 28 * 28),  # Fully connected layer with 28*28 neurons (to reconstruct the image).
            nn.Sigmoid()  # Sigmoid activation function to ensure the output is in the range [0, 1].
        )

    def forward(self, x):
        """
        Forward pass of the autoencoder.
        """
        # Compress the input image to a lower-dimensional representation and then reconstruct it back to the original image.
        encoded_image = self.encoder(x)  # Pass the input through the encoder.
        # Reconstruct the image from the encoded representation.
        reconstructed_image = self.decoder(encoded_image)  # Pass the encoded representation through the decoder.

        return reconstructed_image.view(-1, 1, 28, 28)  # Return the reconstructed image in its original shape (28x28).